In [1]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizerFast,DistilBertForSequenceClassification,TrainingArguments,Trainer
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")+"/IMDB Dataset.csv"

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [4]:
df = pd.read_csv(path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df['label'] = df['sentiment'].map({'positive':1,'negative':0})
df = df[['review','label']]
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds,test_ds

(Dataset({
     features: ['review', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['review', 'label'],
     num_rows: 10000
 }))

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
  return tokenizer(batch['review'],padding='max_length',truncation=True,max_length=256)

train_ds = train_ds.map(tokenize,batched=True)
test_ds = test_ds.map(tokenize,batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
print('Column after tokenization')
print(train_ds.column_names)

Column after tokenization
['review', 'label', 'input_ids', 'attention_mask']


In [8]:
print('First tokenized example')
print(train_ds[0])

First tokenized example
{'review': 'I caught this little gem totally by accident back in 1980 or \'81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, within seconds, the audience was in hysterics! The biggest laugh came when they showed "Princess Laia" having huge cinnamon buns instead of hair on her head. She looks at the camera, gives a grim smile and nods. That made it even funnier! You gotta see "Chewabacca" played by what looks like a Muppet! It was extremely silly and stupid...but I couldn\'t stop laughing. Most of the dialogue was drowned out because of all the laughter. Also if you know "Star Wars" pretty well it\'s even funnier--they deliberately poke fun at some of the dialogue. This REALLY works with an audience! A definite 10!', 'label': 1, 'input_ids': [101, 1045, 3236, 2023, 2210, 17070, 6135

In [9]:
cols = ['input_ids','attention_mask','label']
train_ds.set_format(type='torch',columns=cols)
test_ds.set_format(type='torch',columns=cols)

In [10]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load('f1')

def compile_metrics(eval_pred):
  logits,labels = eval_pred
  preds = np.argmax(logits,axis=-1)
  return {
      'accuracy':accuracy.compute(predictions=preds,references=labels)['accuracy'],
      'precision':precision.compute(predictions=preds,references=labels)['precision'],
      'recall':recall.compute(predictions=preds,references=labels)['recall'],
      'f1':f1.compute(predictions=preds,references=labels)['f1']
  }

In [12]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100
)

In [13]:
from transformers import Trainer

In [14]:
!pip install --upgrade datasets torchvision -q

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    compute_metrics=compile_metrics
)
trainer.train()
trainer.evaluate()

In [16]:
model.save_pretrained('distilbert_imdb')
tokenizer.save_pretrained('distilbert_imdb')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_imdb/tokenizer_config.json', 'distilbert_imdb/tokenizer.json')

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'sentiment-analysis',
    model='distilbert_imdb',
    tokenizer='distilbert_imdb'
)

print(classifier('this movie was absolutely fantastic'))
print(classifier('worst movie ever made.'))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.5570002794265747}]
[{'label': 'LABEL_0', 'score': 0.5640200972557068}]
